## Load data

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

sys.path.append(str(Path().resolve().parent))

In [ ]:
from src.data.load_data import load_data

train, test = load_data()

## Fill MV

In [ ]:
from src.features.missing_values import process_missing_values
from src.features.missing_values import has_missing_values

for col in train.columns:
    if col in test.columns:
        if pd.api.types.is_integer_dtype(train[col].dtype):
            # use nullable Int64 to support NaN
            test[col] = test[col].astype('Int64')
        else:
            test[col] = test[col].astype(train[col].dtype)

train = process_missing_values(train.copy())
print("train have mv? - ", has_missing_values(train))

print()

test = process_missing_values(test.copy())
print("test have mv? - ", has_missing_values(test))

## Feature engineering

### Drop outliers in train

In [ ]:
from src.features.build_features import drop_outliers

train = drop_outliers(train)

### Apply feature engineering

In [ ]:
from src.features.build_features import apply_feature_engineering

[train, log_transformed_features, new_cyclic_numeric_features] = apply_feature_engineering(train)
test = apply_feature_engineering(test)

## Prepare for modeling

### Create target and inputs

In [ ]:
y_train = train['SalePrice']
x_train = train.drop(columns=['SalePrice', 'Id'])
print(x_train.shape)
print(y_train.shape)

In [ ]:
x_test = test.drop(columns=['Id'])
print(x_test.shape)

### Transforming pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

standard_scaler_candidates = log_transformed_features + new_cyclic_numeric_features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), standard_scaler_candidates)
    ],
    remainder='passthrough'
)

### KFold CV

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

### Evaluation strategy

- Primary metric: RMSE
- Target transformation: log1p(SalePrice)
- Evaluation method: 5-fold cross-validation
- Model selection based on CV RMSE

In [ ]:
from sklearn.metrics import root_mean_squared_error

def rmse(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred)

## Modeling

In [ ]:
from sklearn.model_selection import cross_val_predict

### Ridge

In [ ]:
from sklearn.linear_model import Ridge

a = 16

# 1. Build and CV
ridge = Ridge(alpha=a)

ridge_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', ridge)
    ]
)

scores_ridge = cross_val_score(
    ridge_pipeline,
    x_train,
    y_train,
    cv=kf,
    scoring='neg_root_mean_squared_error'
)

ridge_rmse = -scores_ridge.mean()
print(f'a = {a}, RMSE = {ridge_rmse}')

- a = 10, RMSE = 0.11368403663171142
- a = 23, RMSE = 0.11363918955659771
- **best** a = 16, RMSE = 0.11355879152123527

In [ ]:
from sklearn.linear_model import Lasso

# alphas = [0.0003, 0.0005, 0.001, 0.003, 0.005, 0.01]
# alpha ↓  ⇒  max_iter ↑

a = 0.0007
mi = 400

# 1. Build and CV
lasso = Lasso(alpha=a, max_iter=mi)

lasso_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', lasso)
    ]
)

scores_lasso = cross_val_score(
    lasso_pipeline,
    x_train,
    y_train,
    cv=kf,
    scoring='neg_root_mean_squared_error'
)

lasso_rmse = -scores_lasso.mean()
print(f'a = {a}, max_iter = {mi}, RMSE = {lasso_rmse}')

- a = 0.0003, max_iter = 1000, RMSE = 0.11398290869652188
- a = 0.001, max_iter = 1000, RMSE = 0.11392730470993288
- a = 0.0008, max_iter = 1000, RMSE = 0.11320699985487193
- a = 0.0006, max_iter = 1000, RMSE = 0.11305758380396222
- a = 0.0007, max_iter = 1000, RMSE = 0.1130480117746929
- **best** a = 0.0007, max_iter = 400, RMSE = 0.1130480117746929

### Build Ridge + Lasso stacking. Evaluate CV RMSE

In [ ]:
oof_ridge = cross_val_predict(ridge_pipeline, x_train, y_train, cv=kf)
oof_lasso = cross_val_predict(lasso_pipeline, x_train, y_train, cv=kf)

Z = np.column_stack([oof_ridge, oof_lasso])

meta_rmse_scores = []
for train_idx, val_idx in kf.split(Z):
    Z_tr, Z_val = Z[train_idx], Z[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    meta_model = Ridge()
    meta_model.fit(Z_tr, y_tr)
    pred = meta_model.predict(Z_val)
    
    rmse = root_mean_squared_error(y_val, pred)
    meta_rmse_scores.append(rmse)

stacking_cv_rmse = np.mean(meta_rmse_scores)
print("Stacking CV RMSE on train: ", stacking_cv_rmse)

final_meta_model = Ridge()
final_meta_model.fit(Z, y_train)

weights = final_meta_model.coef_
intercept = final_meta_model.intercept_

print("Meta-model weights:")
print(f"Ridge weight: {weights[0]}")
print(f"Lasso weight: {weights[1]}")
print(f"Intercept: {intercept}")

# TODO

1. rewrite MissingValue and FeatureEngineering using Pipeline and transformer
2. apply feature engineering on test.csv
3. train Lasso on whole train.csv
4. predict on test.csv
5. np.expm1 on predictions

In [ ]:
lasso_pipeline.fit(x_train, y_train)

y_pred_log = lasso_pipeline.predict(x_test)

y_pred = np.expm1(y_pred_log)

print(y_pred)